# CYMEK — CS-TRANSFER-001
## Physical class-space causal transfer: V4096 vs V24576

This notebook is an **operator only**. It detaches the exact scientific executable
`a916d1c8d2637abb86d16b1c78e418c95461f3c7` and never executes the moving branch head as science.

**Question:** with byte-identical token sequences and matched shared initialization,
does changing only the physical tied vocabulary/output geometry from 24,576 rows to
4,096 rows improve identity/copy formation at 8L/256w?

Eight mandatory arms: 4 matched seeds × 2 physical vocabularies.  
Each arm: 480 updates / 1,966,080 real tokens.  
Drive root: `/content/drive/MyDrive/CYMEK/CS_TRANSFER_001`.

Do not delete or edit partial Drive state. Re-running this notebook is a resume operation.


In [ ]:
# 1 — Exact executable checkout + static/CPU qualification
import os, sys, json, subprocess, pathlib, hashlib, time

SCIENCE = "a916d1c8d2637abb86d16b1c78e418c95461f3c7"
BRANCH = "cymek-cs-transfer-001"
REPO = pathlib.Path("/content/An-Ra-the-new-AGI-cs-transfer-001")
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", SCIENCE, "--depth", "1"], check=True)
else:
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", SCIENCE, "--depth", "1"], check=True)

subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SCIENCE], check=True)
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert HEAD == SCIENCE, (HEAD, SCIENCE)
print("SCIENTIFIC EXECUTABLE:", HEAD)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "pytest"], check=True)

EXPECTED_BLOBS = {
    "anra_v5/cs_transfer_001_run_v3.py": "3ecdcf317a3e3fe22398d0302ccd4193e09f110c",
    "anra_v5/cs_transfer_001_run_v2.py": "5f372b21358e1eb6fe8a8cae2b16b9c5aada6020",
    "anra_v5/cs_transfer_001_run.py": "ee60921f2bf9b4465b16d4637338f1dcec9c4865",
    "anra_v5/cs_transfer_001_data.py": "b53b7d1ab9661853fbce5a5860882f6e0cdea48c",
    "anra_v5/cs_transfer_001_model.py": "35806598fc9119cd76a88ea265d38a716437062c",
    "experiments/CS_TRANSFER_001/PREREGISTRATION.json": "6f2fb6210adf3dd5fb5dfeac40260fdd12562214",
    "experiments/CS_TRANSFER_001/AMENDMENT_1.json": "55c1f5dff370c06f5d6b9f31bf92222e2c12a983",
    "tools/validate_cs_transfer_001.py": "26339b9ef5d4599acb92327ea170bad64e45ae99",
    "tests/test_cs_transfer_001.py": "ebd5d25962efa6a0d9a3ddb6d67d49843b76b6d6",
    "tests/test_cs_transfer_001_runner.py": "e7962a2a1ea4010817f8bc9d19bb449cd3bf565d",
    "tests/test_cs_transfer_001_protocol.py": "f0fd8833b2b624eecde4b2f2e891b8300a1c9e53",
    "v5_model/core.py": "7cf64b6f557a0556c074e5f61adfc86702f4c725",
    "v5_training/production_backend.py": "72646373c1890e19deaeef634b3f4a0dbdf99632",
    "v5_training/checkpoint.py": "6bd1d04dad43e402b9acff9128d1ad56528a37d0",
    "v5_training/step.py": "bf1d0411be249d2e6e4f8634381124e4f6c7e92f",
    "v5_data/corpus_loading.py": "2c5e64656d9241d2c916549c9fb938e1df848ee0",
}
for path, expected in EXPECTED_BLOBS.items():
    got = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", f"HEAD:{path}"], text=True).strip()
    assert got == expected, f"blob drift {path}: {got} != {expected}"
print("CRITICAL BLOBS: PASS", len(EXPECTED_BLOBS))

subprocess.run([sys.executable, "tools/validate_cs_transfer_001.py"], cwd=REPO, check=True)
TESTS = [
    "tests/test_cs_transfer_001.py",
    "tests/test_cs_transfer_001_runner.py",
    "tests/test_cs_transfer_001_protocol.py",
    "tests/test_v5_production_backend.py",
    "tests/test_v5_checkpoint_adapter.py",
]
for t in TESTS:
    print("\n>>>", t)
    subprocess.run([sys.executable, "-m", "pytest", t, "-q", "--tb=short"], cwd=REPO, check=True)
print("\nCPU QUALIFICATION: PASS")


In [ ]:
# 2 — Mount Drive and bind the dedicated persistent root BEFORE runner imports
from google.colab import drive
drive.mount("/content/drive")

ROOT = pathlib.Path("/content/drive/MyDrive/CYMEK/CS_TRANSFER_001")
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["CS_TRANSFER_001_ROOT"] = str(ROOT)
assert pathlib.Path(os.environ["CS_TRANSFER_001_ROOT"]).resolve() == ROOT.resolve()

def run_json(args, *, stream=False):
    cmd = [sys.executable, "-m", "anra_v5.cs_transfer_001_run_v3", *args]
    print("\n$", " ".join(cmd))
    if stream:
        p = subprocess.Popen(cmd, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             bufsize=1, env=os.environ.copy())
        lines = []
        for line in p.stdout:
            print(line, end="")
            lines.append(line)
        rc = p.wait()
        if rc:
            raise RuntimeError(f"command failed rc={rc}")
        text = "".join(lines)
    else:
        p = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True, env=os.environ.copy())
        print(p.stdout, end="")
        if p.stderr.strip():
            print(p.stderr, file=sys.stderr)
        if p.returncode:
            raise RuntimeError(f"command failed rc={p.returncode}")
        text = p.stdout
    start = text.rfind("\n{")
    candidate = text[start+1:] if start >= 0 else text[text.find("{"):]
    try:
        return json.loads(candidate)
    except Exception:
        return {"raw_stdout": text}

protocol = run_json(["--mode", "protocol"])
assert protocol.get("amendment_status") == "PROSPECTIVE_PREEXECUTION"
assert protocol.get("prompt_suffix") == "\n"
assert protocol.get("answer_prefix") == ""
print("EFFECTIVE PROTOCOL:", protocol.get("effective_protocol_sha256"))

data_receipt = ROOT / "receipts" / "DATA.json"
if not data_receipt.exists():
    prep = run_json(["--mode", "prepare"], stream=True)
else:
    print("DATA receipt already exists; preserving Drive state and validating via scan/preflight.")

d = json.loads(data_receipt.read_text())
surface = d["shared_surface"]
assert surface["contamination"]["clean"] is True
assert surface["predictive_shortcut_max"] < 0.35
assert d["maximum_allowed_content_id"] == 4095
assert d["identical_token_sequences_required"] is True
for split, fams in surface["acceptance"].items():
    for fam, row in fams.items():
        assert float(row["acceptance_rate"]) >= 0.15, (split, fam, row)
for split, stats in surface["token_stats"].items():
    assert stats["all_lt_4096"] is True
    assert int(stats["maximum_content_id"]) < 4096
print("DATA GATES: PASS")
print("Drive root:", ROOT)


In [ ]:
# 3 — Real CUDA preflight on BOTH physical arms; no scientific checkpoints
import torch, gc
assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
pre = run_json(["--mode", "preflight", "--cuda"], stream=True)
print("CUDA PREFLIGHT COMPLETE")
runs_root = ROOT / "runs"
if runs_root.exists():
    bad = list(runs_root.glob("pair_*/*/state/*/LATEST"))
    assert not bad, f"preflight contaminated scientific state: {bad}"
print("PREFLIGHT ISOLATION: PASS")
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 4 — Scan current persistent state
scan = run_json(["--mode", "scan"])
print(json.dumps(scan, indent=2))
action = scan.get("safe_action") or scan.get("SAFE_ACTION") or scan.get("action")
if action is not None:
    assert "FAIL_CLOSED" not in str(action), scan


In [ ]:
# 5 — Reach the fixed endpoint for all 8 mandatory arms (safe to re-run after disconnect)
ARMS = [(p, a) for p in range(4) for a in ("PHYS_4096", "PHYS_24576")]
for pair, arm in ARMS:
    print("\n" + "="*72)
    print(f"PAIR {pair} / {arm}")
    print("="*72)
    result = run_json(["--mode", "run-arm", "--pair-index", str(pair), "--arm", arm, "--cuda"], stream=True)
    s = run_json(["--mode", "scan"])
    print("SCAN AFTER ARM:", json.dumps(s, indent=2)[:4000])
print("\nALL ARM COMMANDS RETURNED. Proceed to development aggregation.")


In [ ]:
# 6 — Freeze development result BEFORE any sealed consumption
dev = run_json(["--mode", "development"], stream=True)
print("\nDEVELOPMENT AGGREGATE FROZEN")
print(json.dumps(dev, indent=2)[:10000])
dev_path = ROOT / "receipts" / "DEVELOPMENT_AGGREGATE.json"
assert dev_path.is_file(), "development aggregate missing"


In [ ]:
# 7 — One-shot sealed finalization
sealed_marker = ROOT / "SEALED_CONSUMPTION.json"
final_path = ROOT / "receipts" / "FINAL_RESULT.json"
if final_path.exists():
    print("FINAL_RESULT already exists; sealed evaluation will NOT be repeated.")
    final = json.loads(final_path.read_text())
elif sealed_marker.exists():
    marker = json.loads(sealed_marker.read_text())
    raise RuntimeError("FAIL_CLOSED: sealed marker exists without FINAL_RESULT. Do not delete it; a fresh sealed identity/amendment is required. Current marker=" + repr(marker))
else:
    final = run_json(["--mode", "finalize", "--cuda"], stream=True)
assert final_path.exists(), "finalizer returned without FINAL_RESULT"
final = json.loads(final_path.read_text())
print("\nFINAL RESULT")
print(json.dumps(final, indent=2)[:20000])


In [ ]:
# 8 — Build compact result bundle (exclude checkpoints and raw split rows)
import zipfile
ZIP = pathlib.Path("/content/CS_TRANSFER_001_RESULTS.zip")
if ZIP.exists(): ZIP.unlink()
included = []
with zipfile.ZipFile(ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(ROOT.rglob("*")):
        if not path.is_file(): continue
        rel = path.relative_to(ROOT)
        parts = set(rel.parts)
        if "state" in parts or "data" in parts: continue
        z.write(path, arcname=str(rel)); included.append(str(rel))
bundle_sha = hashlib.sha256(ZIP.read_bytes()).hexdigest()
print("BUNDLE:", ZIP)
print("FILES:", len(included))
print("SHA256:", bundle_sha)
print("SIZE MiB:", round(ZIP.stat().st_size/2**20, 2))
print("\nUpload this ZIP back to ChatGPT for interpretation.")
